# Jueces open-weight (v3): jurado QLoRA + Qwen 2.5 7B + Llama 3.1 8B sobre las 18.000 citas

Este cuaderno implementa el **sistema de jueces** de la actividad A2 del enunciado: varios modelos abiertos clasifican la misma cita y la pre-etiqueta sale del voto mayoritario.

- **Juez 1:** Qwen 2.5 7B ajustado con QLoRA (columna de `ANOTACION_18000_CITAS_REALES_QLORA.csv`; ninguna de las 18.000 citas estuvo en su entrenamiento).
- **Juez 2:** `Qwen/Qwen2.5-7B-Instruct`
- **Juez 3:** `meta-llama/Llama-3.1-8B-Instruct`
- **Candidatos probados y descartados en la prueba de 200 citas:** `mistralai/Mistral-7B-Instruct-v0.3`, Granite 4.2 8B y OLMo 3 7B.

Cada juez elige **una letra** (9 funciones + Ambiguo + No evaluable). Se leen las probabilidades de las 11 letras en una sola pasada, sin generar texto.

**Historia de versiones (3-nov-2026):**
- **v1 (descartada):** prompt con definiciones cortas y un solo orden de opciones. κ de Fleiss −0,014. El diagnóstico mostró que Llama elegía por posición (al invertir las opciones cambiaba de etiqueta en 15 de 22 citas) y que Mistral llamaba casi todo *Comparison* o *Evidence*.
- **v2 (descartada tras la prueba de 200):** definiciones y reglas de la guía de anotación, `Answer:` escrito al inicio de la respuesta, Qwen 2.5 7B como tercer juez y dos órdenes de opciones (normal e invertido). κ de Fleiss 0,224, pero Llama y Mistral pusieron casi la mitad en *Improvement / Modification*: con 11 opciones, la del centro (letra F) **no se mueve al invertir el orden**, y Llama prefiere las letras del medio.
- **v3 (este cuaderno):** igual que la v2, pero cada cita se clasifica con **tres rotaciones** de las opciones, así cada clase pasa por el inicio, el medio y el final de la lista. En una GPU de 40 GB el modelo se carga **completo en bf16** (sin cuantizar). Se guardan las probabilidades de cada rotación y el sesgo de cada letra con una entrada vacía, para poder comparar formas de agregarlas sin volver a usar la GPU.

**Jurado escogido (prueba de 200 citas, 3-nov-2026):** contra el consenso de dos referencias comerciales (Claude Sonnet 5.5 y GPT-6.1 Sol coinciden en 164 citas, κ 0,757 entre ellas), el QLoRA acierta 77 %, Qwen 2.5 7B 65 %, Llama 3.1 8B 43 %, Granite 4.2 8B 24 %, OLMo 3 7B 18 % y Mistral 7B 12 %. El jurado queda en **QLoRA + Qwen + Llama**: cuando los tres coinciden aciertan 93 % (55 citas) y cuando coinciden dos, 71 % (82 citas). Mistral, Granite y OLMo se descartan. Las referencias comerciales solo sirvieron para escoger el método; no son etiquetas humanas.

Todo se guarda en **Google Drive** mientras corre, con el sufijo de versión (`_v3`). Si Colab se desconecta, al volver a ejecutar la celda el juez sigue donde iba.

**Importante:** son pre-etiquetas de modelos. No reemplazan la anotación humana del Test Gold.

## Paso 0. Lo que necesitas antes de empezar

1. **Colab con GPU:** menú *Entorno de ejecución → Cambiar tipo de entorno → A100* (Colab Pro). En la T4 gratuita también corre, pero mucho más lento.
2. **Licencia de Llama** aceptada en https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct (es "gated"). Mistral v0.3 y Qwen 2.5 7B son Apache-2.0 y no piden aprobación.
3. **Token de Hugging Face** de tipo *Read* en los Secretos de Colab, con el nombre `HF_TOKEN`. **No pegues el token en una celda.**
4. **El CSV en Drive:** `ANOTACION_18000_CITAS_REALES_QLORA.csv` en `MyDrive/proyecto_grado/jueces/`.

5. **Solo para las referencias (Pasos 4c y 4d):** llaves de las APIs en los Secretos de Colab con los nombres `ANTHROPIC_API_KEY` y `OPENAI_API_KEY`.

**Tiempo:** en la prueba de la v2 (A100, dos órdenes, 4 bits) cada juez procesó 7-9 citas por segundo. La v3 hace tres pasadas por cita, así que se espera algo más lento (estimado, no medido).

**Orden de trabajo:** Pasos 0 a 3 → Paso 4 (prueba de 200 citas) → Pasos 4c y 4d (referencias con Sonnet y GPT) → Paso 4e (modelos candidatos) → Paso 4f (comparación) → **revisar** → Paso 5 (corrida completa) → Paso 6 (jurado).

In [ ]:
# Comprobar que hay GPU asignada (debe aparecer una NVIDIA, por ejemplo A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Instalar las librerías que no trae Colab: bitsandbytes cuantiza el modelo a 4 bits para que quepa en la GPU
!pip install -q -U transformers accelerate bitsandbytes

## Paso 1. Configuración y datos

In [ ]:
import os, json, time, math, hashlib, random                         # utilidades estándar
from collections import Counter                                      # conteos de etiquetas
from pathlib import Path                                             # rutas multiplataforma

import pandas as pd                                                  # lectura y escritura del CSV
from google.colab import drive                                       # acceso a Google Drive desde Colab

drive.mount('/content/drive')                                        # monta Drive; pide autorización la primera vez

CARPETA_DRIVE = Path('/content/drive/MyDrive/proyecto_grado/jueces')  # carpeta donde están el CSV y donde se guardan los resultados
CSV_ENTRADA   = CARPETA_DRIVE / 'ANOTACION_18000_CITAS_REALES_QLORA.csv'  # las 18.000 citas con la predicción QLoRA
CARPETA_DRIVE.mkdir(parents=True, exist_ok=True)                     # crea la carpeta si no existe

VERSION = 'v3'           # va en el nombre de cada archivo; si se cambia el prompt o el método, súbela (v4, v5…)

JUECES = {                                                           # jueces open-weight zero-shot que votan (junto al QLoRA)
    'llama':   'meta-llama/Llama-3.1-8B-Instruct',
    'qwen':    'Qwen/Qwen2.5-7B-Instruct',
}
CANDIDATOS = {                                                       # probados en las 200 citas y descartados (ver Paso 4f)
    'mistral': 'mistralai/Mistral-7B-Instruct-v0.3',
    'granite': 'ibm-granite/granite-4.2-8b',
    'olmo':    'allenai/Olmo-3-7B-Instruct',
}
MODELOS = {**JUECES, **CANDIDATOS}                                   # todos los que correr_juez sabe cargar

RESUMEN_CHARS = 400      # se recorta el resumen del artículo citado (mismo valor que en el cuaderno QLoRA)
LOTE          = 8        # citas por pasada (cada una va en tres rotaciones); si aparece "CUDA out of memory", bájalo a 4
SEMILLA       = 42       # se fija para que la prueba corta tome siempre las mismas citas

assert CSV_ENTRADA.exists(), f'No encuentro {CSV_ENTRADA}: súbelo a Drive o corrige CARPETA_DRIVE'

In [ ]:
# Leer el CSV (utf-8-sig quita la marca BOM que deja Excel)
df = pd.read_csv(CSV_ENTRADA, encoding='utf-8-sig', dtype=str, keep_default_na=False)

COL = {'id': 'context_id', 'ctx': 'Contexto TARGETCIT', 'titulo': 'Título citado',
       'resumen': 'Resumen citado', 'ev': '¿Evaluable?', 'fn': 'Función principal', 'just': 'Justificación / duda'}
faltan = [c for c in COL.values() if c not in df.columns]           # comprobar que el archivo es el esperado
assert not faltan, f'Faltan columnas: {faltan}'
assert df[COL['id']].is_unique, 'Hay context_id repetidos'          # cada cita debe aparecer una sola vez

print(len(df), 'citas')                                              # deben ser 18.000
print('Predicción QLoRA (función):', df[COL['fn']].value_counts().to_dict())

## Paso 2. Taxonomía y prompt

Las definiciones y las reglas de desempate vienen de la guía de anotación del equipo (`docs/guia_anotacion_funcion_cita.md`), traducidas al inglés. El experto lo planteó así: "yo le doy los criterios al modelo del lenguaje y el modelo me va a dar una categoría". El prompt sigue siendo zero-shot (sin ejemplos), así que los jueces no copian el criterio del QLoRA ni de Haiku.

Las reglas nombran las clases, no las letras, para que valgan igual con las opciones en cualquier orden.

**Rotaciones:** con la rotación 0 la lista va de A) Background a K) No evaluable; con la rotación 4 empieza en *Application*, y con la 8 en *Further Reading*. Así ninguna clase queda siempre en la misma letra.

In [ ]:
OPCIONES = [   # (etiqueta, definición de la guía de anotación)
    ('Background', 'the citation gives general context, summarizes prior literature, describes the state of a field or gives historical background'),
    ('Gap', 'the citation points out something not yet done, a limitation of previous work, or a gap that motivates the current study'),
    ('Basis', 'the cited work is a direct intellectual or conceptual foundation of the current research, which is built on it'),
    ('Comparison', 'the text explicitly compares methods, results, data, theories or approaches between the current work and the cited work, or among cited works'),
    ('Application', 'the current work directly uses a method, algorithm, tool, dataset or procedure from the cited work without substantial modification'),
    ('Improvement / Modification', 'the current work adapts, extends, modifies or improves a method or tool from the cited work for a new goal or condition'),
    ('Evidence', 'the citation supports a specific claim, hypothesis, methodological decision, figure or interpretation in the citing text'),
    ('Identification of the Originator', 'the main function is to credit who originally introduced an idea, method, concept or theory'),
    ('Further Reading', 'the citation sends the reader to a source for additional details that the current text does not develop'),
    ('Ambiguo', 'ambiguous: two or more readings are equally plausible, or the truncated context does not allow a decision'),
    ('No evaluable', 'not evaluable: the context gives no information about the function of this citation'),
]
ETIQUETAS = [e for e, _ in OPCIONES]                                 # 11 etiquetas en orden fijo (orden de las columnas de salida)
LETRAS = 'ABCDEFGHIJK'                                               # una letra por opción
assert len(LETRAS) == len(ETIQUETAS)

REGLAS = [     # reglas para casos ambiguos de la guía de anotación
    'A list or enumeration of previous works is usually Background. Choose Comparison only when there is an explicit contrast of similarities, differences, performance or properties.',
    'Background places the work within the literature; Evidence supports one specific claim. Ask whether the sentence would still be a general description of the field.',
    'Basis is a conceptual dependency of the current work; Application is direct practical use of a method, algorithm, tool or data.',
    'If the cited method is used as it is, choose Application; if it is adapted, extended or modified, choose Improvement / Modification.',
    'Do not choose Identification of the Originator just because a reference is old; there must be an explicit attribution of origin (e.g. "introduced", "first proposed", "originally developed").',
    'Do not choose Further Reading just because "see" appears; the citation must mainly send the reader to additional details.',
]

def prompt_usuario(fila, opciones=OPCIONES):
    """Arma el mensaje del usuario para una cita, con las opciones en el orden dado."""
    resumen = fila[COL['resumen']][:RESUMEN_CHARS]                  # el resumen se recorta para no alargar el prompt
    lineas = [
        'Classify the rhetorical function of the citation marked TARGETCIT in a computational linguistics paper.',
        'OTHERCIT marks other citations. The context may be truncated.',
        '',
        f'Cited paper title: {fila[COL["titulo"]]}',
        f'Cited paper abstract (truncated): {resumen}',
        f'Citation context: {fila[COL["ctx"]]}',
        '',
        'Options:',
    ]
    lineas += [f'{l}) {e} - {d}' for l, (e, d) in zip(LETRAS, opciones)]   # lista de opciones con su letra
    lineas += ['', 'Rules for difficult cases:']
    lineas += [f'- {r}' for r in REGLAS]
    lineas += ['', 'Answer with a single letter from A to K.']
    return '\n'.join(lineas)

ROTACIONES = (0, 4, 8)                                               # con estas tres, cada clase cae en una letra del inicio, una del medio y una del final
ORDENES = [OPCIONES[s:] + OPCIONES[:s] for s in ROTACIONES]          # la rotación s pone en la letra A la opción número s
PREFIJO = 'Answer:'                                                  # inicio de la respuesta que se deja escrito
FILAS_VACIAS = [{COL['titulo']: v, COL['resumen']: v, COL['ctx']: v} for v in ('N/A', '', '[MASK]')]   # entradas sin contenido: miden el sesgo de cada letra

def posiciones(opciones):
    """Para cada etiqueta (en el orden de ETIQUETAS), la letra que le tocó en `opciones`."""
    orden = [e for e, _ in opciones]
    return [orden.index(e) for e in ETIQUETAS]

HASH_PROMPT = hashlib.sha256(prompt_usuario(df.iloc[0]).encode()).hexdigest()[:12]   # huella para documentar qué prompt se usó
print(prompt_usuario(df.iloc[0]))
print('\nhuella del prompt:', HASH_PROMPT)

## Paso 3. Funciones del juez, del jurado y del acuerdo

In [ ]:
import gc                                                            # liberar memoria entre un juez y otro
import torch                                                         # PyTorch ya viene en Colab
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))                                # el token se lee del secreto, nunca se escribe en el cuaderno

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16   # la T4 no soporta bfloat16; usa float16
MEMORIA_GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
CUANTIZAR = MEMORIA_GPU_GB < 30                                      # A100 (40 GB): modelo completo en bf16; T4 (15 GB): cuantizado a 4 bits
print(torch.cuda.get_device_name(0), f'{MEMORIA_GPU_GB:.0f} GB ->', 'cuantizado a 4 bits' if CUANTIZAR else 'precisión completa')

def cargar(modelo_id):
    """Carga el modelo (completo o cuantizado a 4 bits, según la GPU) y su tokenizador."""
    tok = AutoTokenizer.from_pretrained(modelo_id)
    tok.padding_side = 'left'                                        # con relleno a la izquierda, la última posición es la respuesta
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token                                # Llama y Mistral no traen token de relleno
    opciones_carga = {'device_map': {'': 0}, 'dtype': DTYPE}
    if CUANTIZAR:
        opciones_carga['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                                  bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
    modelo = AutoModelForCausalLM.from_pretrained(modelo_id, **opciones_carga)
    modelo.eval()
    # Cada letra puede tokenizarse sola ("A") o con espacio delante (" A"); se suman ambas variantes
    ids_letra = []
    for l in LETRAS:
        variantes = {tok.encode(l, add_special_tokens=False)[-1], tok.encode(' ' + l, add_special_tokens=False)[-1]}
        ids_letra.append(sorted(variantes))
    return tok, modelo, ids_letra

def texto_chat(tok, fila, opciones=OPCIONES, prefijo=PREFIJO):
    """Aplica la plantilla de chat del modelo y deja escrito el inicio de la respuesta."""
    mensajes = [{'role': 'user', 'content': prompt_usuario(fila, opciones)}]
    return tok.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True) + prefijo

@torch.no_grad()
def ultimo_logit(modelo, enc):
    """Logits de la última posición: ahí el modelo escribiría la letra."""
    try:
        return modelo(**enc, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()   # solo el último token: ahorra memoria
    except TypeError:
        return modelo(**enc, use_cache=False).logits[:, -1, :].float()                     # versiones antiguas de transformers

@torch.no_grad()
def probs_letras(tok, modelo, ids_letra, textos):
    """Probabilidad de cada una de las 11 letras para cada texto (en el orden de las letras A-K)."""
    enc = tok(textos, return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
    logits = ultimo_logit(modelo, enc)
    puntajes = torch.stack([torch.logsumexp(logits[:, ids], dim=-1) for ids in ids_letra], dim=-1)
    return puntajes.softmax(-1).cpu()

def sesgo_letras(tok, modelo, ids_letra):
    """Para cada rotación, probabilidad media de cada letra cuando la cita viene vacía (sesgo del modelo hacia ciertas letras)."""
    return [probs_letras(tok, modelo, ids_letra, [texto_chat(tok, v, opciones) for v in FILAS_VACIAS]).mean(0)
            for opciones in ORDENES]

@torch.no_grad()
def predecir(tok, modelo, ids_letra, filas):
    """Probabilidad de cada etiqueta promediada sobre las tres rotaciones (cada clase pasa por varias letras)."""
    acumulado = torch.zeros(len(filas), len(ETIQUETAS))
    por_rotacion = []
    for opciones in ORDENES:
        p_letra = probs_letras(tok, modelo, ids_letra, [texto_chat(tok, f, opciones) for f in filas])
        por_rotacion.append(p_letra)
        acumulado += p_letra[:, posiciones(opciones)]                # misma probabilidad, reordenada por etiqueta
    probs = acumulado / len(ORDENES)
    salida = []
    for i, p in enumerate(probs):
        k = int(p.argmax())
        elegidas = [ORDENES[r][int(pr[i].argmax())][0] for r, pr in enumerate(por_rotacion)]   # etiqueta ganadora en cada rotación
        salida.append({'etiqueta': ETIQUETAS[k], 'p': round(float(p[k]), 4),
                       'estable': all(e == ETIQUETAS[k] for e in elegidas),   # True si ganó la misma etiqueta en las tres rotaciones
                       'dist': [round(float(x), 4) for x in p],
                       'p_rotaciones': [[round(float(x), 4) for x in pr[i]] for pr in por_rotacion]})   # por letra, para analizar después
    return salida

def leer_hechos(ruta):
    """Lee las predicciones ya guardadas para poder reanudar."""
    hechos = {}
    if ruta.exists():
        for linea in ruta.read_text(encoding='utf-8').splitlines():
            if linea.strip():
                d = json.loads(linea)
                hechos[d['context_id']] = d
    return hechos

def ruta_juez(nombre):
    return CARPETA_DRIVE / f'juez_{nombre}_{VERSION}.jsonl'

def correr_juez(nombre, solo_ids=None):
    """Clasifica las citas pendientes con un juez y las va guardando en Drive (archivo JSONL).
    Con `solo_ids` se limita a esas citas (la prueba corta); si ya están hechas, no procesa nada."""
    ruta = ruta_juez(nombre)
    hechos = leer_hechos(ruta)
    pendientes = df[~df[COL['id']].isin(hechos)].copy()
    if solo_ids is not None:
        pendientes = pendientes[pendientes[COL['id']].isin(solo_ids)]
    pendientes['_largo'] = pendientes[COL['ctx']].str.len() + pendientes[COL['resumen']].str.len().clip(upper=RESUMEN_CHARS)
    pendientes = pendientes.sort_values('_largo')                    # lotes de largo parecido = menos relleno y más velocidad
    print(f'{nombre} ({VERSION}): {len(hechos)} ya hechas, {len(pendientes)} por procesar')
    if len(pendientes) == 0:
        return hechos
    tok, modelo, ids_letra = cargar(MODELOS[nombre])
    sesgo = sesgo_letras(tok, modelo, ids_letra)                     # se guarda en el meta para analizar el sesgo de posición
    filas = [f for _, f in pendientes.iterrows()]
    t0 = time.time()
    with open(ruta, 'a', encoding='utf-8') as salida:
        for i in range(0, len(filas), LOTE):
            lote = filas[i:i + LOTE]
            for f, r in zip(lote, predecir(tok, modelo, ids_letra, lote)):
                registro = {'context_id': f[COL['id']], 'juez': nombre, **r}
                salida.write(json.dumps(registro, ensure_ascii=False) + '\n')
                hechos[registro['context_id']] = registro
            salida.flush()                                           # se escribe en Drive en cada lote
            hechas = i + len(lote)
            if hechas % (LOTE * 50) < LOTE or hechas == len(filas):
                vel = hechas / (time.time() - t0)
                print(f'  {hechas}/{len(filas)}  {vel:.1f} filas/s  faltan ~{(len(filas) - hechas) / vel / 60:.0f} min')
    meta = {'juez': nombre, 'version': VERSION, 'modelo': MODELOS[nombre], 'hash_prompt': HASH_PROMPT,
            'rotaciones': list(ROTACIONES), 'prefijo': PREFIJO,
            'sesgo_letras_entrada_vacia': [[round(float(x), 4) for x in s] for s in sesgo],
            'cuantizacion': 'nf4 4-bit' if CUANTIZAR else 'ninguna (bf16 completo)',
            'dtype': str(DTYPE), 'filas': len(hechos), 'fecha': time.strftime('%Y-%m-%d %H:%M'),
            'transformers': __import__('transformers').__version__, 'torch': torch.__version__,
            'gpu': torch.cuda.get_device_name(0)}
    (CARPETA_DRIVE / f'juez_{nombre}_{VERSION}_meta.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    del modelo                                                       # libera la GPU para el siguiente juez
    gc.collect()
    torch.cuda.empty_cache()
    return hechos

In [ ]:
def voto_mayoritario(etiquetas):
    """Etiqueta elegida por al menos 2 jueces; si todas difieren, 'SIN_CONSENSO'."""
    etiqueta, votos = Counter(etiquetas).most_common(1)[0]
    return (etiqueta, votos) if votos >= 2 else ('SIN_CONSENSO', 1)

def kappa_cohen(a, b):
    """Acuerdo entre dos jueces corregido por azar."""
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    ca, cb = Counter(a), Counter(b)
    pe = sum(ca[k] * cb[k] for k in set(ca) | set(cb)) / (n * n)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0

def kappa_fleiss(listas):
    """Acuerdo de varios jueces corregido por azar; listas = una lista de etiquetas por juez."""
    m = len(listas)                                                  # número de jueces
    n = len(listas[0])                                               # número de citas
    categorias = sorted({e for l in listas for e in l})
    total_cat = Counter()
    p_i = 0.0
    for i in range(n):
        c = Counter(l[i] for l in listas)
        total_cat.update(c)
        p_i += (sum(v * v for v in c.values()) - m) / (m * (m - 1))
    p_bar = p_i / n
    p_e = sum((total_cat[k] / (n * m)) ** 2 for k in categorias)
    return (p_bar - p_e) / (1 - p_e) if p_e < 1 else 1.0

def etiqueta_qlora(f):
    """Convierte las columnas del CSV QLoRA a una de las 11 etiquetas."""
    return f[COL['fn']] if f[COL['ev']] == 'Evaluable' else f[COL['ev']]

def particion(cid):
    """Partición de ACL-200 según el año del artículo citante (P13-... = 2013, W99-... = 1999)."""
    a = int(cid[1:3])                                                # dos dígitos del año
    if a == 15:
        return 'test'
    if a == 14:
        return 'val'
    if a <= 13:
        return 'train'                                               # 2000-2013
    return 'anterior_2000'                                           # 1980-1999: aparte hasta confirmar cómo los trata ACL-200

QLORA = {f[COL['id']]: etiqueta_qlora(f) for _, f in df.iterrows()}  # etiqueta del QLoRA por cita

import re                                                            # expresiones regulares para leer respuestas de texto

def etiqueta_de(texto):
    """Lee la respuesta FINAL de un LLM comercial (suele explicar primero). Devuelve (etiqueta, cómo se leyó) o (None, 'error')."""
    lineas = [l.strip().strip('*#').strip() for l in texto.strip().splitlines() if l.strip()]
    if lineas:                                                       # 1) la última línea es la letra: "E", "**E**", "Answer: E", "E) Application"
        ultima = re.sub(r'^(?:[Ff]inal [Aa]nswer|[Aa]nswer)\s*[:\-]?\s*', '', lineas[-1]).strip('*() .').strip()
        m = re.match(r'^([A-K])(?=[).:\s]|$)', ultima)
        if m and not re.match(r'^A\s+[a-z]', ultima):                # pero no el artículo "A" de una frase ("A list of…")
            return OPCIONES[LETRAS.index(m.group(1))][0], 'letra_final'
    marcas = re.findall(r'(?:[Aa]nswer(?: is)?\s*[:\-]?\s*\**|\*\*)\(?([A-K])\)?(?=[\s.*):]|$)', texto)
    if marcas:                                                       # 2) la última letra marcada: "**E**", "the answer is E"
        return OPCIONES[LETRAS.index(marcas[-1])][0], 'letra_marcada'
    cola = texto[-150:].lower()                                      # 3) una sola clase nombrada al final ("…so it is Background.")
    nombradas = {e for e in ETIQUETAS if re.search(r'\b' + re.escape(e.lower()) + r'\b', cola)}
    if len(nombradas) == 1:
        return nombradas.pop(), 'nombre_final'
    return None, 'error'

def leer_referencia(ruta):
    """Etiquetas válidas de un archivo de referencia (Sonnet o GPT), sin los errores de formato."""
    return {cid: d['etiqueta'] for cid, d in leer_hechos(ruta).items() if d['etiqueta'] != 'ERROR_FORMATO'}

def comparar_referencias(etiquetas, refs, ids):
    """Acierto de cada modelo contra cada referencia y contra el consenso de las referencias, y valor de los tríos QLoRA + Qwen + X.
    etiquetas = {modelo: {cita: etiqueta}}; refs = {referencia: {cita: etiqueta}}."""
    nombres = list(refs)
    if len(nombres) == 2:
        a, b = nombres
        comunes = [i for i in ids if i in refs[a] and i in refs[b]]
        print(f'{a} vs {b}: coinciden en {sum(refs[a][i] == refs[b][i] for i in comunes) / len(comunes):.0%} '
              f'de {len(comunes)} citas | κ {kappa_cohen([refs[a][i] for i in comunes], [refs[b][i] for i in comunes]):.3f}')
    consenso = [i for i in ids if all(i in r for r in refs.values()) and len({r[i] for r in refs.values()}) == 1]
    ref_c = {i: refs[nombres[0]][i] for i in consenso}
    print(f'Referencia de consenso (todas las referencias coinciden): {len(consenso)} citas\n')
    filas = []
    for m, et in etiquetas.items():
        partes = []
        for nr in nombres + ['consenso']:
            r = ref_c if nr == 'consenso' else refs[nr]
            c = [i for i in ids if i in r and i in et]
            partes.append(sum(et[i] == r[i] for i in c) / len(c) if c else float('nan'))
        filas.append((m, partes))
    print(f'{"modelo":10s}' + ''.join(f'{"vs " + n:>14s}' for n in nombres + ['consenso']))
    for m, partes in sorted(filas, key=lambda x: -x[1][-1]):
        print(f'{m:10s}' + ''.join(f'{p:>14.0%}' for p in partes))
    print()
    for x in [m for m in etiquetas if m not in ('qlora', 'qwen')]:
        niveles = {}
        for i in consenso:
            if all(i in etiquetas[m] for m in ('qlora', 'qwen', x)):
                e, v = Counter([etiquetas['qlora'][i], etiquetas['qwen'][i], etiquetas[x][i]]).most_common(1)[0]
                niveles.setdefault(v, []).append(e == ref_c[i])
        total = sum(len(g) for g in niveles.values())
        con = niveles.get(3, []) + niveles.get(2, [])
        texto = ' | '.join(f'{v}/3: {len(niveles[v])} citas, {sum(niveles[v]) / len(niveles[v]):.0%}' for v in sorted(niveles, reverse=True) if v >= 2)
        print(f'trío qlora+qwen+{x}: consenso en {len(con)}/{total}, acierto con consenso {sum(con) / max(len(con), 1):.0%} ({texto})')
    return consenso

def resumen_acuerdo(resultados):
    """Imprime distribución, estabilidad y acuerdo de los jueces sobre las citas que todos tienen."""
    ids = sorted(set.intersection(*[set(r) for r in resultados.values()]))
    etiquetas = {n: [r[i]['etiqueta'] for i in ids] for n, r in resultados.items()}
    print(f'{len(ids)} citas con todos los jueces')
    for n, r in resultados.items():
        estables = sum(r[i].get('estable', False) for i in ids) / len(ids)
        con_qlora = sum(r[i]['etiqueta'] == QLORA[i] for i in ids) / len(ids)
        print(f'\n{n}: misma etiqueta en las tres rotaciones {estables:.0%} | coincide con QLoRA {con_qlora:.0%}')
        print('   ', Counter(etiquetas[n]).most_common())
    nombres = list(etiquetas)
    if len(nombres) >= 3:
        print(f'\nκ de Fleiss ({", ".join(nombres)}): {kappa_fleiss([etiquetas[n] for n in nombres]):.3f}')
    for i, a in enumerate(nombres):
        for b in nombres[i + 1:]:
            print(f'κ de Cohen {a}-{b}: {kappa_cohen(etiquetas[a], etiquetas[b]):.3f}')
        print(f'κ de Cohen {a}-qlora: {kappa_cohen(etiquetas[a], [QLORA[i] for i in ids]):.3f}')
    if len(nombres) >= 3:
        votos = [voto_mayoritario([etiquetas[n][k] for n in nombres]) for k in range(len(ids))]
        print('\nJueces de acuerdo:', dict(Counter(v for _, v in votos)))
        print('Voto:', Counter(e for e, _ in votos).most_common())
    return ids

## Paso 4. Prueba corta (200 citas por juez) y revisión

Las mismas 200 citas pasan por los tres jueces. **Antes de seguir al Paso 5, revisa la salida** (o mándasela a quien te esté ayudando):

- que ningún juez ponga casi todo en una o dos clases;
- que la mayoría de citas tenga la misma etiqueta en las tres rotaciones;
- que el κ entre jueces sea claramente mayor que 0.

Si algo de eso falla, **no corras el Paso 5**: se ajusta el método y se sube `VERSION`, para no mezclar resultados de métodos distintos. Las 200 citas de la prueba quedan guardadas y el Paso 5 sigue desde ahí. Son las mismas 200 citas de la prueba v2, así que las dos versiones se pueden comparar.

El diagnóstico de la v1 está en el cuaderno `jueces_llama_mistral_v1.ipynb` y en `diagnostico_jueces.csv` (Drive).

In [ ]:
MUESTRA = set(df.sample(n=200, random_state=SEMILLA)[COL['id']])     # siempre las mismas 200 citas (las de la prueba v2)
completos_prueba = {n: correr_juez(n, solo_ids=MUESTRA) for n in JUECES}   # si ya están hechas, no vuelve a procesarlas
prueba = {n: {i: r[i] for i in MUESTRA if i in r} for n, r in completos_prueba.items()}
ids_prueba = resumen_acuerdo(prueba)

## Paso 4c (opcional). Referencia con Claude Sonnet sobre las mismas 200 citas

Para saber cuál juez acierta más hace falta una referencia que no sea uno de ellos. Aquí Claude Sonnet clasifica las mismas 200 citas con **el mismo prompt** (rotación 0). Es solo una referencia de desarrollo para escoger el método de los jueces: **no es una etiqueta humana** y no entra al jurado ni al Test Gold. También deja los primeros datos de costo, latencia y errores de formato para la actividad A7.

Necesita la llave en los Secretos de Colab con el nombre `ANTHROPIC_API_KEY`. Si se corta, al repetir la celda sigue donde iba.

**Corrida r1 (3-nov-2026, descartada):** Sonnet explica primero y pone la letra al final; la celda guardaba solo 200 caracteres y buscaba la letra al inicio, así que 175 de 200 respuestas quedaron como error de formato. Costó US$0,73 (219.155 tokens de entrada y 28.182 de salida) con 2,56 s de latencia media. La **r2** guarda la respuesta completa, lee la respuesta final y registra cómo la leyó.

In [ ]:
!pip install -q anthropic
import anthropic

MODELO_REFERENCIA = 'claude-sonnet-5-5'
cliente = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))   # la llave se lee del secreto
ruta_ref = CARPETA_DRIVE / 'referencia_sonnet_200_r2.jsonl'          # r1 guardó la respuesta recortada y se descartó
hechas_ref = leer_hechos(ruta_ref)                                   # si ya están las 200, no vuelve a llamar a la API

with open(ruta_ref, 'a', encoding='utf-8') as salida:
    for cid in ids_prueba:
        if cid in hechas_ref:
            continue
        f = df[df[COL['id']] == cid].iloc[0]
        t0 = time.time()
        r = cliente.messages.create(model=MODELO_REFERENCIA, max_tokens=2048,
                                    messages=[{'role': 'user', 'content': prompt_usuario(f)}])
        texto = ''.join(b.text for b in r.content if b.type == 'text')       # solo los bloques de texto (no el razonamiento)
        etiqueta, lectura = etiqueta_de(texto)
        registro = {'context_id': cid, 'etiqueta': etiqueta or 'ERROR_FORMATO', 'lectura': lectura,
                    'respuesta': texto, 'stop_reason': r.stop_reason, 'latencia_s': round(time.time() - t0, 2),   # respuesta completa
                    'tokens_entrada': r.usage.input_tokens, 'tokens_salida': r.usage.output_tokens,
                    'modelo': MODELO_REFERENCIA, 'hash_prompt': HASH_PROMPT}
        salida.write(json.dumps(registro, ensure_ascii=False) + '\n')
        salida.flush()
        hechas_ref[cid] = registro

ref = {cid: hechas_ref[cid]['etiqueta'] for cid in ids_prueba}
print('Cómo se leyó la respuesta:', dict(Counter(hechas_ref[c]['lectura'] for c in ids_prueba)))
print('Errores de formato:', sum(e == 'ERROR_FORMATO' for e in ref.values()))
print('Tokens: entrada', sum(hechas_ref[c]['tokens_entrada'] for c in ids_prueba), '| salida', sum(hechas_ref[c]['tokens_salida'] for c in ids_prueba))
print('Latencia media (s):', round(sum(hechas_ref[c]['latencia_s'] for c in ids_prueba) / len(ids_prueba), 2))
print('Sonnet:', Counter(ref.values()).most_common())
for n, r in prueba.items():
    et = [r[c]['etiqueta'] for c in ids_prueba]
    print(f'{n}: coincide con Sonnet {sum(a == ref[c] for a, c in zip(et, ids_prueba)) / len(ids_prueba):.0%} | κ {kappa_cohen(et, [ref[c] for c in ids_prueba]):.3f}')
print(f'qlora: coincide con Sonnet {sum(QLORA[c] == ref[c] for c in ids_prueba) / len(ids_prueba):.0%} | κ {kappa_cohen([QLORA[c] for c in ids_prueba], [ref[c] for c in ids_prueba]):.3f}')

## Paso 4d. Segunda referencia con GPT sobre las mismas 200 citas

Sonnet y Haiku son de la misma empresa (Anthropic), y el QLoRA aprendió de etiquetas de Haiku. Para descartar que el QLoRA salga favorecido solo por "pensar parecido" a Sonnet, GPT clasifica las mismas 200 citas con **el mismo prompt**. Donde Sonnet y GPT coinciden queda una referencia de consenso de dos proveedores. Sigue siendo una referencia de desarrollo, no una etiqueta humana.

Modelo: `gpt-6.1-sol`, que OpenAI recomienda para equilibrar capacidad y costo (US$2 por millón de tokens de entrada y US$10 de salida, verificado el 3-nov-2026). Necesita `OPENAI_API_KEY` en los Secretos de Colab. Si se corta, al repetir la celda sigue donde iba.

In [ ]:
!pip install -q openai
from openai import OpenAI

MODELO_GPT = 'gpt-6.1-sol'
cliente_gpt = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))         # la llave se lee del secreto
ruta_gpt = CARPETA_DRIVE / 'referencia_gpt_200.jsonl'
hechas_gpt = leer_hechos(ruta_gpt)

with open(ruta_gpt, 'a', encoding='utf-8') as salida:
    for cid in ids_prueba:
        if cid in hechas_gpt:
            continue
        f = df[df[COL['id']] == cid].iloc[0]
        t0 = time.time()
        r = cliente_gpt.responses.create(model=MODELO_GPT, input=prompt_usuario(f), max_output_tokens=4096)
        texto = r.output_text or ''                                  # texto final (sin el razonamiento interno)
        etiqueta, lectura = etiqueta_de(texto)
        uso = getattr(r, 'usage', None)                              # se lee con getattr por si cambian los nombres de los campos
        registro = {'context_id': cid, 'etiqueta': etiqueta or 'ERROR_FORMATO', 'lectura': lectura,
                    'respuesta': texto, 'estado': getattr(r, 'status', None), 'latencia_s': round(time.time() - t0, 2),
                    'tokens_entrada': getattr(uso, 'input_tokens', None), 'tokens_salida': getattr(uso, 'output_tokens', None),
                    'modelo': MODELO_GPT, 'hash_prompt': HASH_PROMPT}
        salida.write(json.dumps(registro, ensure_ascii=False) + '\n')
        salida.flush()
        hechas_gpt[cid] = registro

g = [hechas_gpt[c] for c in ids_prueba]
print('Cómo se leyó la respuesta:', dict(Counter(x['lectura'] for x in g)))
print('Estado de las respuestas:', dict(Counter(x['estado'] for x in g)))
print('Errores de formato:', sum(x['etiqueta'] == 'ERROR_FORMATO' for x in g))
print('Tokens: entrada', sum(x['tokens_entrada'] or 0 for x in g), '| salida', sum(x['tokens_salida'] or 0 for x in g))
print('Latencia media (s):', round(sum(x['latencia_s'] for x in g) / len(g), 2))
print('GPT:', Counter(x['etiqueta'] for x in g).most_common())

## Paso 4e. Modelos candidatos para reemplazar a Mistral

Mistral 7B quedó al nivel del azar contra Sonnet (11 % de acierto, κ 0,06). Aquí se prueban dos modelos abiertos de otras familias, sobre las mismas 200 citas y con el mismo método: **Granite 4.2 8B** (IBM, Apache-2.0) y **OLMo 3 7B Instruct** (AI2, Apache-2.0, con datos de entrenamiento abiertos). Ninguno pide aprobación de licencia. Si alguno no carga, se informa y se sigue con el otro.

In [ ]:
for n in CANDIDATOS:
    try:
        completos_prueba[n] = correr_juez(n, solo_ids=MUESTRA)
        prueba[n] = {i: completos_prueba[n][i] for i in MUESTRA if i in completos_prueba[n]}
    except Exception as e:                                           # un candidato que no carga no detiene la prueba
        print(f'{n}: no se pudo correr ({type(e).__name__}: {e})')
ids_prueba = resumen_acuerdo({n: r for n, r in prueba.items() if len(r) == len(MUESTRA)})

## Paso 4f. Comparación contra las referencias

Muestra el acierto de cada modelo contra Sonnet, contra GPT y contra el consenso de los dos, y cuánto acierta cada trío QLoRA + Qwen + X según cuántos jueces coinciden. **Copia toda la salida** para escoger el tercer juez.

In [ ]:
etiquetas = {'qlora': {i: QLORA[i] for i in ids_prueba}}
for n, r in prueba.items():
    etiquetas[n] = {i: r[i]['etiqueta'] for i in ids_prueba if i in r}
refs = {'sonnet': leer_referencia(CARPETA_DRIVE / 'referencia_sonnet_200_r2.jsonl'),
        'gpt': leer_referencia(CARPETA_DRIVE / 'referencia_gpt_200.jsonl')}
consenso_ref = comparar_referencias(etiquetas, refs, ids_prueba)

## Paso 5. Corrida completa

**Solo después de revisar el Paso 4.** Si Colab se desconecta, vuelve a montar Drive (Paso 1), ejecuta los Pasos 2 y 3, y repite la celda del juez que iba: continúa donde quedó.

In [ ]:
llama = correr_juez('llama')
print(len(llama), 'de', len(df))

In [ ]:
qwen = correr_juez('qwen')
print(len(qwen), 'de', len(df))

## Paso 6. Jurado: voto mayoritario y acuerdo entre jueces

Votan tres jueces open-weight: el **QLoRA** (Qwen 2.5 7B ajustado con etiquetas de Haiku) y dos en zero-shot, **Qwen 2.5 7B** y **Llama 3.1 8B**. La etiqueta final es la que eligen al menos dos de los tres; si los tres difieren, queda `SIN_CONSENSO`. La columna `jueces_de_acuerdo` (3 o 2) es la confianza de la pre-etiqueta: en la prueba de 200, con 3/3 se acertó 93 % y con 2/3, 71 %.

In [ ]:
ABIERTOS = ['qwen', 'llama']                                         # jueces zero-shot que votan junto al QLoRA
VOTANTES = ['qlora'] + ABIERTOS

def armar_jurado(completos, ids):
    """Una fila por cita: etiqueta de cada votante, voto mayoritario, nivel de acuerdo y partición."""
    filas = []
    for cid in ids:
        etiquetas = [QLORA[cid]] + [completos[n][cid]['etiqueta'] for n in ABIERTOS]
        voto, n_votos = voto_mayoritario(etiquetas)
        fila = {'context_id': cid, 'particion': particion(cid), 'voto': voto, 'jueces_de_acuerdo': n_votos, 'qlora': QLORA[cid]}
        for n in ABIERTOS:
            fila[n] = completos[n][cid]['etiqueta']
            fila[f'p_{n}'] = completos[n][cid]['p']
            fila[f'estable_{n}'] = completos[n][cid]['estable']
        filas.append(fila)
    return filas

completos = {n: leer_hechos(ruta_juez(n)) for n in ABIERTOS}
ids = sorted(set(QLORA) & set.intersection(*[set(c) for c in completos.values()]))
print(f'{len(ids)} citas con los tres votantes (deben ser 18.000)')
filas = armar_jurado(completos, ids)

listas = {v: [f[v] for f in filas] for v in VOTANTES}
print(f"κ de Fleiss ({', '.join(VOTANTES)}): {kappa_fleiss(list(listas.values())):.3f}")
for i, a in enumerate(VOTANTES):
    for b in VOTANTES[i + 1:]:
        print(f'κ de Cohen {a}-{b}: {kappa_cohen(listas[a], listas[b]):.3f}')
for n in ABIERTOS:
    print(f"{n}: misma etiqueta en las tres rotaciones {sum(f['estable_' + n] for f in filas) / len(filas):.0%}")

jurado = pd.DataFrame(filas)
print('\nJueces de acuerdo:', jurado.jueces_de_acuerdo.value_counts().to_dict())
print('\nVoto por clase y partición:')
print(pd.crosstab(jurado.voto, jurado.particion, margins=True))
print('\nCitas con acuerdo 3/3 por clase:', jurado[jurado.jueces_de_acuerdo == 3].voto.value_counts().to_dict())

salida = CARPETA_DRIVE / f'jurado_18000_{VERSION}.csv'
jurado.to_csv(salida, index=False, encoding='utf-8-sig')               # resultado final en Drive
print('Guardado en', salida)

## Qué queda en Drive

- `juez_<juez>_v3.jsonl`: una línea por cita con la etiqueta, su probabilidad, si fue estable en las tres rotaciones, la distribución promediada y las probabilidades de cada rotación.
- `juez_<juez>_v3_meta.json`: modelo, huella del prompt, rotaciones, sesgo de cada letra con entrada vacía, versiones y GPU usadas (para reproducir).
- `jurado_18000_v3.csv`: las etiquetas del QLoRA, Qwen y Llama, el voto, el nivel de acuerdo y la partición de cada cita.
- `juez_granite_v3.jsonl`, `juez_olmo_v3.jsonl` (Paso 4e): los candidatos sobre las 200 citas de prueba.
- `referencia_gpt_200.jsonl` (Paso 4d): lo mismo que la de Sonnet, con GPT-6.1 Sol.
- `referencia_sonnet_200_r2.jsonl` (si se corrió el Paso 4c): etiqueta de referencia, respuesta completa, forma de lectura, tokens y latencia de cada cita de la prueba. La r1 (`referencia_sonnet_200.jsonl`) se conserva como evidencia del error de lectura.

**Siguiente paso (fuera de este cuaderno):** muestrear el Test Gold desde la partición `test`, al azar por categoría según el voto, excluyendo los contextos ya expuestos. Según la guía de anotación, cada anotador decide **sin ver** la pre-etiqueta; la pre-etiqueta solo se usa para escoger las citas y en la reconciliación. Las citas de `test` **no** se usan para entrenar.